# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs using the Croissant metadata.

In [ ]:
# List all available record sets and their @id
print("Available record sets (by @id):\n")
record_sets = metadata.record_sets
for rs in record_sets:
    print(f"@id: {rs.id} | name: {rs.name if hasattr(rs, 'name') else ''}")
    # List available fields/columns
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for field in rs.fields:
            print(f"    @id: {field.id} | {field.name if hasattr(field, 'name') else ''}")
    if hasattr(rs, 'columns') and rs.columns:
        print("  Columns:")
        for column in rs.columns:
            print(f"    @id: {column.id} | {column.name if hasattr(column, 'name') else ''}")
    print("-")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# ---
# Example extraction using the available record set @id(s):
record_set_ids = [rs.id for rs in metadata.record_sets]
print("All available record set @ids:", record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for {record_set_id}: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records found for record set {record_set_id}.")

# For further analysis, select a non-empty dataframe
if dataframes:
    main_record_set_id = next(iter(dataframes.keys()))
else:
    main_record_set_id = None
print(f"Selected record set for analysis: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section can include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Replace with an appropriate numeric field @id from the overview (see "Columns for ..." above)
numeric_field = None
group_field = None
if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Try finding a numeric field automatically
    for col in df.columns:
        # Try to convert column to numeric, skip if fails
        try:
            series = pd.to_numeric(df[col], errors='coerce')
            if series.notna().sum() > 0:
                numeric_field = col
                break
        except Exception:
            pass
    print(f"Selected numeric field: {numeric_field}")

    # Example filter and normalization
    if numeric_field:
        df_num = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df_num.mean() if not pd.isnull(df_num.mean()) else 0
        filtered_df = df[df_num > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean):")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (df_num - df_num.mean()) / df_num.std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by another field if available
        other_fields = [c for c in df.columns if c != numeric_field]
        if other_fields:
            group_field = other_fields[0]  # Just pick the first
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
else:
    print("No usable data loaded in previous step.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if main_record_set_id and numeric_field:
    plt.figure(figsize=(7,4))
    data = pd.to_numeric(dataframes[main_record_set_id][numeric_field], errors='coerce')
    data = data.dropna()
    plt.hist(data, bins=30, alpha=0.7)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Optionally, plot against the group field if available
    if group_field:
        plt.figure(figsize=(10,4))
        grouped = dataframes[main_record_set_id].groupby(group_field)[numeric_field].mean()
        grouped = pd.to_numeric(grouped, errors='coerce').dropna()
        grouped.plot(kind='bar')
        plt.title(f"Average {numeric_field} by {group_field}")
        plt.ylabel(numeric_field)
        plt.xlabel(group_field)
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

<!-- Use this cell to write your summary: what were the main features of the dataset; were there any outliers or unusual patterns; what could future work or analysis focus on? -->